[< Back to Main README](../README.md) | [Demo README](./README.md)

# Hooks vs Agent Control — Block vs Self-Correct

## The Problem with Blocking

Hooks (functions that intercept agent actions) enforce business rules at the tool level — when a rule fails, `cancel_tool` blocks the call and the agent reports failure. The user then has to rephrase or adjust their request manually.

However, in many scenarios the agent can self-correct the issue rather than blocking the request entirely. If the user asks for 15 guests and the max is 10 per room, the agent could split the booking into two rooms and complete the reservation — instead of just saying "I can't do that."

## The Solution: Steer Instead of Block

[Agent Control](https://github.com/agentcontrol/agent-control) introduces **steer controls** — when a violation is detected, the agent receives corrective guidance via `Guide()` and retries with the fix applied:

| | Hooks | Agent Control |
|---|---|---|
| Where rules live | Python code (`hooks=[...]`) | Server — API/dashboard |
| When rule fails | `cancel_tool = "BLOCKED"` → agent fails | `Guide("split into 2 rooms")` → agent retries corrected |
| To change a rule | Edit code, redeploy | API call — no code changes |
| Integration | `HookProvider` + `hooks=[...]` | `Plugin` + `plugins=[...]` |

## The Tools

Three booking tools in `tools.py` — clean, no validation logic:

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `book_hotel(hotel, check_in, check_out, guests)` | Books a hotel room | Returns `"SUCCESS: Booking BK001..."` — no guest limit in the tool |
| `process_payment(amount, booking_id)` | Processes payment | Returns `"SUCCESS"` or `"ERROR: Booking not found"` |
| `confirm_booking(booking_id)` | Confirms a booking | Returns `"SUCCESS: Confirmed BK001"` |

The tools do NOT enforce the max-guests rule. That is the guardrail layer's job.

## What We Test

Same query, same tools, same model — only the guardrail changes:

| Test | Guardrail | Expected behavior |
|------|-----------|-------------------|
| 1 — Hooks | `MaxGuestsHook` with `cancel_tool` | Agent is BLOCKED, asks user what to do |
| 2 — Agent Control | `AgentControlSteeringHandler` with `Guide()` | Agent self-corrects — splits into 2 rooms (10 + 5 guests) |

This notebook compares two guardrail approaches for AI agents: **Hooks** (block invalid operations) vs **Agent Control** (steer the agent to self-correct). Both approaches are tested on the same scenario — booking a hotel with 15 guests when the maximum is 10.

## Prerequisites

**1. AWS credentials.** This demo runs on Amazon Bedrock through the Strands default model, the same as demos 02, 03 and 04. No model-provider API key of any kind is required.

**2. An Agent Control server** — required for Test 2 only. It is not bundled with this workshop and not part of `agent-control-sdk`. See the [Demo README](./README.md) for setup. Test 1 runs without it.

To use a different model provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os

if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# Verify AWS credentials are available
import boto3

sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("✅ AWS credentials configured")

# AWS credentials are the ONLY credential this demo needs. Both tests run on
# Amazon Bedrock through the Strands default model, exactly like demos 02, 03
# and 04. No model-provider API key is required.


## Setup

In [ ]:
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

os.environ.setdefault("OTEL_SDK_DISABLED", "true")

from dotenv import load_dotenv
from strands import Agent
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent

from tools import STATE, ALL_TOOLS, reset_state

load_dotenv()

# No `model=` anywhere in this notebook. Both tests use the Strands default, which
# resolves to Amazon Bedrock — the same default demos 02, 03 and 04 use. This is
# what lets the comparison honestly claim "same tools, same model, same query".

# Dates are computed relative to today so a fixed date cannot rot into the past.
CHECK_IN = (datetime.now() + timedelta(days=30)).strftime("%Y-%m-%d")
CHECK_OUT = (datetime.now() + timedelta(days=32)).strftime("%Y-%m-%d")
QUERY = f"Book AnyCompany Lisbon Resort for 15 guests from {CHECK_IN} to {CHECK_OUT}"

PROMPT = (
    "You are a hotel booking assistant. "
    "When booking, first describe what you will book (hotel, guests, dates) "
    "then call the tool."
)

CONTROLS_FILE = Path("controls.yaml").resolve()
SERVER_URL = os.getenv("AGENT_CONTROL_URL", "http://127.0.0.1:8000")

# Cap on how many times one agent invocation may be steered.
#
# The steer control matches a guest count above 10 in LLM output. The agent's own
# corrective reply usually restates the total ("splitting your 15 guests across 2
# rooms"), which re-matches and re-fires the control. Agent Control's regex
# evaluator runs on RE2, which has no lookahead, so the pattern cannot be written
# to exclude the correction. Bounding the retries is the fix.
#
# Left unbounded this livelocks: the agent is steered over and over until the model
# reads the repeated injections as a prompt-injection attack, ignores them, and
# books all 15 guests in one room — the exact outcome the control exists to prevent.
MAX_STEERS = 1


def bookings_created():
    """Bookings made during this run, excluding the pre-seeded BK001 fixture."""
    return [b for bid, b in STATE["bookings"].items() if bid != "BK001"]


print("Setup complete!")


---
## Test 1 — Hooks (Block with cancel_tool)

`MaxGuestsHook` intercepts `BeforeToolCallEvent` and checks the `guests` parameter. If > 10, it sets `event.cancel_tool` — the agent receives this as the tool result and must report the failure to the user.

**Query:** *"Book AnyCompany Lisbon Resort for 15 guests"*. Check-in and check-out are computed relative to today, so the scenario never rots into the past.

> The agent will be blocked and ask the user what to do — the booking will NOT complete.

In [ ]:
class MaxGuestsHook(HookProvider):
    def __init__(self):
        self.blocked = 0

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "book_hotel":
            return
        guests = event.tool_use["input"].get("guests", 1)
        if guests > 10:
            self.blocked += 1
            event.cancel_tool = f"BLOCKED: {guests} guests exceeds maximum of 10"


hook = MaxGuestsHook()
agent_hooks = Agent(system_prompt=PROMPT, tools=ALL_TOOLS, hooks=[hook])

reset_state()  # isolate this test from any earlier run
start = time.time()
response_hooks = agent_hooks(QUERY)
time_hooks = time.time() - start

# Assert on the booking ledger, not on the model's wording.
over_limit_hooks = [b for b in bookings_created() if b["guests"] > 10]
outcome_hooks = "bypassed" if over_limit_hooks else ("blocked" if hook.blocked else "completed")

print(f"Time: {time_hooks:.1f}s")
print(f"Hook blocked: {hook.blocked} call(s)")
print(f"Bookings created: {len(bookings_created())}")
print(f"Outcome: {outcome_hooks}")

if response_hooks.metrics:
    usage = response_hooks.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")


---
## Test 2 — Agent Control (Steer with Guide)

Agent Control evaluates the LLM output before tool execution. When the model says "I will book for 15 guests", the steer control detects the number > 10 and returns a `Guide` with corrective instructions. The agent retries, this time splitting the booking into two rooms.

**Same query, same tools, same model as Test 1** — the only difference is the guardrail layer.

> ### Hard prerequisite: an Agent Control server
>
> This cell needs a running [Agent Control server](https://github.com/agentcontrol/agent-control). It is **not** bundled with this workshop and **not** part of `agent-control-sdk`. Start it, then run `uv run setup_controls.py` to register the controls. Verify with `curl 127.0.0.1:8000/health`.
>
> If your server is not on the default address, set `AGENT_CONTROL_URL` before running.
>
> **Without a server this cell stops with a clear message rather than a stack trace.** Test 1 above needs no server and still runs.

### How Steer Works

1. LLM generates: *"I will book AnyCompany Lisbon Resort for 15 guests..."*
2. `AgentControlSteeringHandler` evaluates LLM output against server controls
3. Regex matches "15 guest" → steer control fires
4. Steering handler returns `Guide("split into 2 rooms: 10 + 5 guests")`
5. LLM retries with guidance → calls `book_hotel(guests=10)` then `book_hotel(guests=5)`
6. Both bookings complete — user is informed the reservation was split across two rooms

### Why steering is bounded

Steering is a retry loop, and retry loops need a stop condition. This control matches a guest count above 10 in the model's output, and the model's corrective reply usually restates the total ("splitting your 15 guests across 2 rooms"), which re-matches and fires the control again. The regex evaluator runs on RE2, which has no lookahead, so the pattern cannot be narrowed to exclude the correction.

Unbounded, that livelocks. Measured behaviour with no cap: the agent was steered 9 times, called `book_hotel` 17 times, then concluded the repeated injected guidance was a prompt-injection attack, said so, ignored it, and booked all 15 guests in a single room — the exact outcome the control existed to prevent. `MAX_STEERS` bounds the loop to one corrective nudge.


In [ ]:
import urllib.error
import urllib.request

import agent_control
from agent_control.integrations.strands import AgentControlPlugin, AgentControlSteeringHandler
from agent_control.control_decorators import ControlViolationError
from strands.experimental.steering import Proceed
from strands.hooks import AfterToolCallEvent


def http_status(url, timeout=3.0):
    """GET url, returning the HTTP status, or None if nothing answered."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.status
    except urllib.error.HTTPError as exc:
        return exc.code
    except (urllib.error.URLError, TimeoutError, OSError):
        return None


def agent_control_server_available(url):
    """True only if a real Agent Control server answers at url.

    Port 8000 is a busy local development port. Checking only that the socket is
    open, or only that /health returns 200, will accept an unrelated service and
    then fail confusingly deep inside the SDK. Agent Control serves its API under
    /api/v1/agents; a foreign service 404s there.
    """
    if http_status(f"{url}/health") != 200:
        return False
    agents_route = http_status(f"{url}/api/v1/agents")
    return agents_route is not None and agents_route != 404


class BoundedSteeringHandler(AgentControlSteeringHandler):
    """Stops steering after MAX_STEERS so an unsatisfiable control cannot livelock."""

    def __init__(self, agent_name, max_steers=MAX_STEERS):
        super().__init__(agent_name=agent_name, enable_logging=False)
        self.max_steers = max_steers
        self.cap_reached = False

    async def steer_after_model(self, **kwargs):
        if self.steers_applied >= self.max_steers:
            self.cap_reached = True
            return Proceed(reason=f"steer cap of {self.max_steers} reached")
        return await super().steer_after_model(**kwargs)


if not agent_control_server_available(SERVER_URL):
    raise RuntimeError(
        f"No Agent Control server at {SERVER_URL}.\n\n"
        "Test 2 steers the agent using controls served by an Agent Control server.\n"
        "That server is a hard prerequisite and is not bundled with this workshop.\n\n"
        "  1. Start it:              https://github.com/agentcontrol/agent-control\n"
        "  2. Register the controls: uv run setup_controls.py\n"
        "  3. Verify it is up:       curl 127.0.0.1:8000/health\n\n"
        "Running elsewhere? Set AGENT_CONTROL_URL before starting the notebook.\n"
        "This demo needs no model-provider API key — AWS credentials are enough."
    )

agent_control.init(
    agent_name="booking-guardrails-demo",
    server_url=SERVER_URL,
    policy_refresh_interval_seconds=0,
)

controls = agent_control.get_server_controls()
if not controls:
    raise RuntimeError(
        f"Connected to {SERVER_URL} but it returned no controls for "
        "'booking-guardrails-demo'. The guardrail layer would be inert and this "
        "test would prove nothing.\n\nRegister them first:  uv run setup_controls.py"
    )
print(f"✅ {len(controls)} control(s) loaded from {SERVER_URL}")

plugin = AgentControlPlugin(
    agent_name="booking-guardrails-demo",
    event_control_list=[BeforeToolCallEvent, AfterToolCallEvent],
    enable_logging=False,
)
steering = BoundedSteeringHandler(agent_name="booking-guardrails-demo")

# No model= — same Strands Bedrock default as Test 1.
agent_ac = Agent(system_prompt=PROMPT, tools=ALL_TOOLS, plugins=[plugin, steering])

reset_state()  # isolate this test from Test 1
start = time.time()
try:
    response_ac = agent_ac(QUERY)
    time_ac = time.time() - start
    denied = None
except ControlViolationError as exc:
    time_ac = time.time() - start
    denied = str(exc)
    response_ac = None

# Assert on the booking ledger, not on the model's wording.
created = bookings_created()
guest_counts = sorted((b["guests"] for b in created), reverse=True)
over_limit = [g for g in guest_counts if g > 10]

if denied:
    outcome_ac = "denied"
elif over_limit:
    outcome_ac = "failed-open"
elif len(created) >= 2 and sum(guest_counts) == 15:
    outcome_ac = "split-bookings"
elif created:
    outcome_ac = "partial"
else:
    outcome_ac = "no-booking"

print(f"Time: {time_ac:.1f}s")
print(f"Steered: {steering.steers_applied} time(s)")
print(f"Bookings created: {len(created)} — guests per booking: {guest_counts}")
if steering.cap_reached:
    print(f"Steer cap of {MAX_STEERS} enforced — control re-matched after the agent had corrected")
print(f"Outcome: {outcome_ac}")

if response_ac is not None and response_ac.metrics:
    usage = response_ac.metrics.accumulated_usage
    print(f"\n💰 Tokens: {usage['inputTokens']} in, {usage['outputTokens']} out, {usage['totalTokens']} total")


In [ ]:
print(f"{'Approach':<35} {'Time':>8} {'Outcome':>20}")
print("-" * 65)
print(f"{'Test 1 - Hooks (cancel_tool)':<35} {time_hooks:>6.1f}s {outcome_hooks:>20}")
print(f"{'Test 2 - Agent Control (steer)':<35} {time_ac:>6.1f}s {outcome_ac:>20}")

print()
print("Key difference:")
print("  Hooks:         agent receives 'BLOCKED' -> asks user what to do")
print("  Agent Control: agent receives Guide('split into rooms') -> books 10 + 5")

# The demo's headline claim, checked against the booking ledger rather than prose.
print()
if outcome_hooks == "blocked" and outcome_ac == "split-bookings":
    print("✅ CLAIM HOLDS — hooks hard-blocked; Agent Control steered the agent to")
    print("   self-correct and complete the booking.")
else:
    print("❌ CLAIM DOES NOT HOLD")
    print("   Expected: hooks='blocked', agent-control='split-bookings'")
    print(f"   Got:      hooks='{outcome_hooks}', agent-control='{outcome_ac}'")


---
## Summary

### When to Use Each Approach

| Approach | Best for |
|----------|----------|
| **Hooks** | Rules that MUST hard-block — no workaround allowed (e.g., payment before confirmation) |
| **Agent Control (steer)** | Rules where the agent CAN self-correct — split bookings, adjust parameters, redact PII |
| **Agent Control (deny)** | Same as hooks but managed on a server — change rules without redeploying code |

### What Just Happened

With **Hooks**, the agent was blocked at 15 guests and could not complete the booking.

With **Agent Control**, the steering rule fired and guided the agent to split the reservation:
- **Room 1:** 10 guests (maximum per room)
- **Room 2:** 5 guests (remaining guests)

The user gets the outcome they wanted — 15 guests accommodated — without any manual intervention.

### Why This Integration Is Simple

Both approaches integrate with a single line change:

```python
# Hooks:
agent = Agent(tools=[...], hooks=[MaxGuestsHook()])

# Agent Control:
agent = Agent(tools=[...], plugins=[AgentControlPlugin(...), AgentControlSteeringHandler(...)])
```

### References

#### Framework Documentation
- [Hooks](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/) — `BeforeToolCallEvent`, `cancel_tool`
- [Steering](https://strandsagents.com/docs/user-guide/concepts/plugins/steering/) — `Guide`, `Proceed`, `SteeringHandler`
- [Agent Control Plugin](https://strandsagents.com/docs/community/plugins/agent-control/) — Integration docs
- [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/) — Swap to Amazon Bedrock, Anthropic, Ollama

#### Guardrail Systems
- [Agent Control GitHub](https://github.com/agentcontrol/agent-control) — Open source, Apache 2.0
- [Agent Control Docs](https://docs.agentcontrol.dev/) — Server setup and API reference

#### Code
- [Code Repository](https://github.com/aws-samples/sample-stop-ai-agent-hallucinations-workshop)